# app

> FastHTML web layer — auth, routes, UI (see `DESIGN.md` §4/§6). Ledger logic lives in `00_core.ipynb`.

In [ ]:
#| default_exp app

In [ ]:
#| export
import os
import threading
import time
from datetime import date, datetime
from pathlib import Path
from urllib.parse import quote
from zoneinfo import ZoneInfo

import uvicorn
from fasthtml.common import *

from habitrack.core import current_streak, load_full, save_full, window

In [ ]:
#| hide
import tempfile
from fasthtml.core import Client

In [ ]:
#| export
DAYS_SHOWN = 10
TZ = ZoneInfo("Europe/Helsinki")  # "today" never depends on server-local time (DESIGN.md §7)
DOW_JA = ["月", "火", "水", "木", "金", "土", "日"]
DEFAULT_ITEMS = ["読書", "運動", "執筆", "瞑想", "早起き"]
CSS = """\
:root{--paper:#EFECE3;--paper-line:#CBCABB;--ink:#23272A;--ink-soft:#5B5F58;--amber:#B9822E;--amber-soft:#E9DCC0}
*{box-sizing:border-box}
body{background:var(--paper);color:var(--ink);font-family:'Space Mono',monospace;
     max-width:520px;margin:0 auto;padding:32px 20px 60px}
h1{font-family:'Shippori Mincho',serif;font-size:22px;margin:0 0 20px;
   border-bottom:2px solid var(--ink);padding-bottom:10px}
table{border-collapse:collapse}
td,th{padding:0;text-align:center}
.item-cell{font-family:'Shippori Mincho',serif;font-size:15px;text-align:left;
           padding-right:12px;white-space:nowrap}
.dow{font-size:9px;color:var(--ink-soft)}
.num{font-size:9px;color:#9a9a8c}
.today .num{color:var(--amber);font-weight:700}
input[type=checkbox]{appearance:none;width:20px;height:20px;border:1.5px solid var(--ink-soft);
  border-radius:3px;cursor:pointer;margin:2px 4px}
input[type=checkbox]:checked{background:var(--ink);border-color:var(--ink)}
input[type=checkbox].today-cb{border-color:var(--amber);border-width:2.5px}
.streak{font-size:10px;font-weight:700;color:var(--amber);background:var(--amber-soft);
        border-radius:3px;padding:1px 5px;margin-left:6px}
.streak.zero{background:none;color:var(--ink-soft)}
input[type=password]{font-family:inherit;font-size:15px;padding:6px 8px;
  border:1.5px solid var(--ink-soft);border-radius:3px;background:none}
button{font-family:inherit;font-size:14px;padding:6px 14px;cursor:pointer;
  border:1.5px solid var(--ink);border-radius:3px;background:var(--ink);color:var(--paper)}"""
HDRS = (
    Link(rel="preconnect", href="https://fonts.googleapis.com"),
    Link(
        href="https://fonts.googleapis.com/css2?family=Shippori+Mincho:wght@500;700&family=Space+Mono:wght@400;700&display=swap",
        rel="stylesheet",
    ),
    Style(CSS),
)

## `toggle_url`

In [ ]:
#| export
def toggle_url(
    item:str,  # item name — percent-encoded so names may contain ? # % & etc.
    day:date,
) -> str:
    "htmx POST target for one checkbox."
    return f"/toggle/{quote(item, safe='')}/{day.isoformat()}"

In [ ]:
assert toggle_url("読書", date(2026, 8, 24)) == "/toggle/%E8%AA%AD%E6%9B%B8/2026-08-24"
assert toggle_url("A?B", date(2026, 8, 24)) == "/toggle/A%3FB/2026-08-24"  # ? must not become a query
assert toggle_url("50%off#tag", date(2026, 8, 24)) == "/toggle/50%25off%23tag/2026-08-24"

## `create_app`

In [ ]:
#| export
def create_app(
    data_path:Path,  # ledger.csv location — outside the static route (DESIGN.md §3)
    password:str,  # login password; must be non-empty
    secret_key:str,  # session cookie signing key; must be non-empty
    https_only:bool=False,  # sess_https_only — production sets this (DESIGN.md §6)
    default_items:list=DEFAULT_ITEMS,  # columns for first-run auto-create
    tz:ZoneInfo=TZ,  # timezone that defines "today"
    static_path:str="static",  # dedicated assets dir; must not contain the ledger
):
    "Build the habitrack FastHTML app; fails fast on empty secrets."
    if not password or not secret_key:
        raise ValueError("password and secret_key must be non-empty")
    expected = password
    lock = threading.Lock()  # serializes toggle read-modify-write (single process)

    def today() -> date:
        return datetime.now(tz).date()

    def auth_before(req, sess):
        if not sess.get("auth"):
            return Redirect("/login")

    app, rt = fast_app(
        before=Beforeware(auth_before, skip=[r"/login"]),
        secret_key=secret_key,
        sess_https_only=https_only,
        static_path=static_path,  # data/ must stay unreachable (DESIGN.md §3)
        hdrs=HDRS,
    )

    def render_table():
        full = load_full(data_path, default_items, today())
        df = window(full, today(), DAYS_SHOWN)
        days = list(df.index)
        t = today()

        header = Tr(
            Th("", cls="item-cell"),
            *[
                Th(Div(DOW_JA[d.weekday()], cls="dow"), Div(str(d.day), cls="num"),
                   cls="today" if d == t else "")
                for d in days
            ],
        )

        rows = [header]
        for item in df.columns:
            streak = current_streak(full, item, today())
            cells = [
                Td(
                    Span(item),
                    Span(str(streak), cls=f"streak{' zero' if streak == 0 else ''}"),
                    cls="item-cell",
                )
            ]
            for d in days:
                cells.append(
                    Td(
                        Input(
                            type="checkbox",
                            checked=bool(df.at[d, item]),
                            cls="today-cb" if d == t else "",
                            hx_post=toggle_url(item, d),
                            hx_target="#ledger",
                            hx_swap="outerHTML",
                            hx_disabled_elt="this",
                        )
                    )
                )
            rows.append(Tr(*cells))

        return Table(*rows, id="ledger")

    @rt("/")
    def get():
        return Titled("習慣ログ", render_table())  # Titled renders the H1 itself

    @rt("/toggle/{item}/{d}")
    def post(item: str, d: str):
        day = date.fromisoformat(d)
        with lock:  # concurrent toggles must not lose writes
            full = load_full(data_path, default_items, today())
            if item in full.columns:
                if day not in full.index:
                    full.loc[day] = False
                full.at[day, item] = not full.at[day, item]
                full.sort_index(inplace=True)
                save_full(full, data_path)
        return render_table()

    @rt("/login", methods=["GET"])
    def login_page():
        return Titled("習慣ログ", Form(
            Input(type="password", name="password", autofocus=True),
            Button("ログイン"),
            method="post", action="/login",
        ))

    @rt("/login", methods=["POST"])
    def login_submit(sess, password: str = ""):
        if password == expected:
            sess["auth"] = True
            return Redirect("/")
        time.sleep(0.5)  # the entire brute-force defense (DESIGN.md §6)
        return Redirect("/login")

    return app

In [ ]:
# empty secrets must be rejected at startup, not silently accepted
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    for pw, key in [("", "k"), ("pw", ""), ("", "")]:
        try:
            create_app(p, pw, key)
            assert False, "empty secret must be rejected"
        except ValueError:
            pass

In [ ]:
# auth flow + toggle roundtrip against a temp ledger
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    cli = Client(create_app(p, "pw", "k"))

    r = cli.get("/")  # unauthenticated -> login
    assert r.status_code == 303 and r.headers["location"] == "/login"
    assert cli.get("/login").status_code == 200

    r = cli.post("/login", data={"password": "wrong"})
    assert r.headers["location"] == "/login"
    r = cli.post("/login", data={"password": "pw"})
    assert r.headers["location"] == "/"

    r = cli.get("/")
    assert r.status_code == 200 and 'id="ledger"' in r.text

    t = datetime.now(TZ).date()
    item = DEFAULT_ITEMS[0]
    assert load_full(p, [], t).at[t, item] == False
    cli.post(toggle_url(item, t))
    assert load_full(p, [], t).at[t, item] == True
    cli.post(toggle_url(item, t))  # toggle back
    assert load_full(p, [], t).at[t, item] == False

In [ ]:
# a hand-added item name with URL metacharacters works end to end
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "ledger.csv"
    p.write_text("date,A?B\n2026-08-23,False\n")
    cli = Client(create_app(p, "pw", "k"))
    cli.post("/login", data={"password": "pw"})
    day = date(2026, 8, 23)
    cli.post(toggle_url("A?B", day))
    assert load_full(p, [], day).at[day, "A?B"] == True
    assert "/toggle/A%3FB/" in cli.get("/").text  # rendered checkboxes carry the encoded URL

In [ ]:
#| eval: false
# Live dev inside Jupyter, fasthtml style: serve and render the app inline
from fasthtml.jupyter import JupyUvi, HTMX

server = JupyUvi(create_app(Path("../data/ledger.csv"), "dev", "dev"), port=8000)
HTMX()  # renders the running app in an iframe
# server.stop()

## `main`

In [ ]:
#| export
def main(
    data_path:Path,  # ledger.csv location
    static_path:str="static",  # passed through to `create_app`
    host:str="127.0.0.1",  # loopback only; Caddy is the sole public path (DESIGN.md §6)
    port:int=5001,
):
    "Entry point: secrets from env (fail-fast on empty), serve on loopback."
    app = create_app(
        data_path,
        password=os.environ.get("HABITRACK_PASSWORD", ""),
        secret_key=os.environ.get("HABITRACK_SECRET_KEY", ""),
        https_only=os.environ.get("HABITRACK_HTTPS_ONLY") == "1",
        static_path=static_path,
    )
    uvicorn.run(app, host=host, port=port)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()